In [1]:
import pandas as pd
topic_greenlist = pd.read_csv("topic_greenlists.csv")
!pip install transformers

In [2]:
topic_greenlist.head()

,topic,token_id,token,similarity
0,technology,806,Ġtechnology,0.6282
1,technology,2903,Ġtech,0.4063
2,technology,3165,Ġtechnical,0.1689
3,technology,3777,ĠTechnology,0.6306
4,technology,3815,ĠNIGHT,0.1886


In [3]:
topic_greenlist["topic"].value_counts()
TOPICS = topic_greenlist["topic"].unique().tolist()
print(TOPICS)

['technology', 'medicine', 'sports', 'politics', 'science', 'entertainment', 'finance', 'history']


## Level 3 - Topic inference via embedding similarity

Map the prompt and each predetermined topic into OPT-2.7b's own embedding space,
then pick the topic with the highest cosine similarity.

- `prompt_vec = mean(embed(tokenize(prompt)))` (mean-pooled token embeddings)
- `topic_vec  = embed(first subtoken of " <topic>")` e.g. `Ġtechnology`
- `score(t)   = cos(prompt_vec, topic_vec)` -> `argmax`

Same geometry that built the greenlists in `04_cosine_similarity.ipynb`.

In [11]:
from huggingface_hub import login
login()

In [12]:
TOPICS = sorted(topic_greenlist["topic"].unique())
print("predetermined topics:", TOPICS)

topic_token_ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in TOPICS]

for t, tid in zip(TOPICS, topic_token_ids):
    print(f"{t!r:14} -> {tokenizer.decode([tid])!r} (id {tid})")

topic_matrix = normed_embeddings[topic_token_ids]

predetermined topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']
'entertainment' -> ' entertainment' (id 4000)
'finance'      -> ' finance' (id 2879)
'history'      -> ' history' (id 750)
'medicine'     -> ' medicine' (id 6150)
'politics'     -> ' politics' (id 2302)
'science'      -> ' science' (id 2866)
'sports'       -> ' sports' (id 1612)
'technology'   -> ' technology' (id 806)


In [ ]:
@torch.no_grad()
def extract_topic(prompt: str) -> str:
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    print(ids)

   
    prompt_vec = normed_embeddings[ids].mean(dim=0)
    prompt_vec = prompt_vec / prompt_vec.norm()

    scores = topic_matrix @ prompt_vec          # (K,) cosine similarity per topic

    ranked = sorted(zip(TOPICS, scores.tolist()), key=lambda x: -x[1])
    for name, s in ranked:
        bar = "#" * max(0, int((s + 1) * 20))   # cosine in [-1, 1] -> 40 char bar
        print(f"  {name:12} {s:+.4f} {bar}")
    return ranked[0][0]

In [16]:
prompts = [
    "The marxits socitey is very revolutionlary",
    "The cricket team won the final match",
    "Scientists discovered a new particle at the collider",
    "Ancient Rome fell after centuries of decline",
    "The new AI chip runs twice as fast",
]

for p in prompts:
    print(f"\n{p!r}")
    print("->", extract_topic(p))


'The marxits socitey is very revolutionlary'
[133, 4401, 1178, 2629, 17380, 1459, 219, 16, 182, 7977, 462, 1766]
  technology   +0.1764 #######################
  history      +0.1672 #######################
  sports       +0.1561 #######################
  science      +0.1348 ######################
  politics     +0.1003 ######################
  medicine     +0.0997 #####################
  entertainment +0.0956 #####################
  finance      +0.0618 #####################
-> technology

'The cricket team won the final match'
[133, 5630, 165, 351, 5, 507, 914]
  sports       +0.2668 #########################
  history      +0.2147 ########################
  science      +0.1877 #######################
  technology   +0.1853 #######################
  politics     +0.1151 ######################
  entertainment +0.1041 ######################
  finance      +0.0982 #####################
  medicine     +0.0955 #####################
-> sports

'Scientists discovered a new particle at th

## First layer - public, topic-based watermark

Mirror of `second_layer.py`'s private scheme, but the greenlist is **public and
fixed per topic** instead of key-derived per context:

| | second layer (private) | first layer (public) |
|---|---|---|
| greenlist | derived from key + prev tokens | fixed per topic (from CSV) |
| selection | hash-based | topic extracted from prompt |
| boost | `scores[b, preferred] += delta_private` | `scores[..., topic_green_ids] += delta` |

Flow: `extract_topic(prompt)` -> assign -> `TopicBoostProcessor` boosts that
topic's greenlist logits at every generation step.

In [17]:
from transformers import LogitsProcessor

class TopicBoostProcessor(LogitsProcessor):
    def __init__(self, green_token_ids, delta=2.0):
        self.ids = torch.tensor(green_token_ids, dtype=torch.long)
        self.delta = delta

    def __call__(self, input_ids, scores):
        scores[..., self.ids.to(scores.device)] += self.delta
        return scores

greenlists = {t: g["token_id"].astype(int).tolist()
              for t, g in topic_greenlist.groupby("topic")}
print("greenlist sizes:", {t: len(v) for t, v in greenlists.items()})

prompt = "The government passed a new law before the election"
assigned = extract_topic(prompt)
print("assigned topic:", assigned)

inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    plain = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=1.0, top_p=0.9)
    boosted = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=1.0, top_p=0.9,
                             logits_processor=[TopicBoostProcessor(greenlists[assigned], delta=4.0)])

print("PLAIN  :", tokenizer.decode(plain[0], skip_special_tokens=True))
print("BOOSTED:", tokenizer.decode(boosted[0], skip_special_tokens=True))

greenlist sizes: {'entertainment': 110, 'finance': 102, 'history': 1670, 'medicine': 95, 'politics': 3566, 'science': 1958, 'sports': 2575, 'technology': 7105}
[133, 168, 1595, 10, 92, 488, 137, 5, 729]
  history      +0.2499 ########################
  technology   +0.2312 ########################
  sports       +0.1985 #######################
  science      +0.1877 #######################
  politics     +0.1722 #######################
  entertainment +0.1165 ######################
  medicine     +0.1006 ######################
  finance      +0.0933 #####################
assigned topic: history
PLAIN  : The government passed a new law before the election that will require all parties to show that they have a "strategic interest" in the project.

The new law aims to protect the environment and is expected to give investors greater confidence in the
BOOSTED: The government passed a new law before the election, giving it the ability to override the courts.       They're using it now becau

In [ ]:
%%writefile first_layer.py
'''First layer: public, topic-based watermarking.

Extracts each prompt's topic (cosine similarity in OPT's own embedding space,
against predetermined topic vectors) and boosts the logits of that topic's
greenlist tokens during generation.
'''
import torch
import pandas as pd

from transformers import LogitsProcessor


def load_topic_greenlists(csv_path):
    df = pd.read_csv(csv_path)
    return {t: g["token_id"].astype(int).tolist() for t, g in df.groupby("topic")}


def build_topic_matrix(model, tokenizer, topics):
    emb = model.get_input_embeddings().weight.detach()
    emb = emb / emb.norm(dim=1, keepdim=True)
    ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in topics]
    return emb[ids]


class TopicBoostProcessor(LogitsProcessor):
    """Adds `delta` to the logits of one topic's greenlist tokens at every step."""

    def __init__(self, green_token_ids, delta=2.0):
        self.green_token_ids = torch.tensor(green_token_ids, dtype=torch.long)
        self.delta = delta

    def __call__(self, input_ids, scores):
        # scores: (batch, vocab) next-token logits
        scores[..., self.green_token_ids.to(scores.device)] += self.delta
        return scores


class TopicWiseWatermarking:
    """Extract topic -> assign -> boost its greenlist tokens while generating."""

    def __init__(self, model, tokenizer, greenlist_csv="topic_greenlists.csv",
                 delta=2.0, temperature=1.0, top_p=0.9, max_new_tokens=50):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.delta = delta
        self.temperature = temperature
        self.top_p = top_p
        self.max_new_tokens = max_new_tokens

        self.greenlists = load_topic_greenlists(greenlist_csv)
        self.topics = sorted(self.greenlists)
        self.topic_matrix = build_topic_matrix(model, tokenizer, self.topics)

    @torch.no_grad()
    def extract_topic(self, prompt):
        """Mean-pool prompt token embeddings, cosine vs each topic vector."""
        ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        vec = self.model.get_input_embeddings().weight[ids].mean(dim=0)
        vec = vec / vec.norm()
        scores = self.topic_matrix @ vec
        ranked = sorted(zip(self.topics, scores.tolist()), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def _generate(self, inputs, processors=None):
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                logits_processor=processors,
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def watermark(self, prompts):
        """Plain vs topic-boosted generation for each prompt. Returns DataFrame."""
        rows = []
        for i, prompt in enumerate(prompts):
            topic, ranked = self.extract_topic(prompt)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            plain = self._generate(inputs)
            watermarked = self._generate(
                inputs,
                processors=[TopicBoostProcessor(self.greenlists[topic], self.delta)],
            )
            rows.append({
                "id": i,
                "prompt": prompt,
                "topic": topic,
                "topic_score": round(ranked[0][1], 4),
                "plain_output": plain,
                "watermarked_output": watermarked,
            })
        return pd.DataFrame(rows)


Overwriting first_layer.py


In [22]:
import sys
sys.path.insert(0, "..")

import first_layer

twm = first_layer.TopicWiseWatermarking(
    model, tokenizer, greenlist_csv="topic_greenlists.csv", delta=4.0
)
results = twm.watermark([
    "The government passed a new law before the election",
    "The cricket team won the final match",
    "The new AI chip runs twice as fast",
])
results

,id,prompt,topic,topic_score,plain_output,watermarked_output
0,0,The government passed a new law before the ele...,history,0.2512,The government passed a new law before the ele...,The government passed a new law before the ele...
1,1,The cricket team won the final match,sports,0.2634,The cricket team won the final match of the se...,The cricket team won the final match of the To...
2,2,The new AI chip runs twice as fast,technology,0.2389,The new AI chip runs twice as fast as a standa...,The new AI chip runs twice as fast as today's ...


In [23]:
results.to_csv("topic_wise_watermarked_example.csv", index=False)